# Seminar: EWMA chart for a mean shift

In this variant you will build an **EWMA** (Exponentially Weighted Moving Average) detector for an upward mean shift.

**What you need to do**
1. Generate a Gaussian series with a jump in the mean.
2. Implement a one-sided EWMA chart.
3. Measure **detection delay** and **time to false alarm (TFA)**.

### The EWMA idea

Instead of judging each observation alone (Shewhart), we keep a smooth running summary:

$$
z_t = \lambda x_t + (1-\lambda) z_{t-1},
\qquad z_0 = \mu_0.
$$

Small $\lambda$ means more memory: past points still matter, so small persistent shifts can accumulate.

For a steady-state one-sided chart we use

$$
\sigma_z = \sigma \sqrt{\frac{\lambda}{2-\lambda}},
\qquad
\mathrm{UCL} = \mu_0 + L\,\sigma_z,
$$

and raise an alarm when

$$
\tau = \min\{ t : z_t > \mathrm{UCL} \}.
$$

## 0. Imports and experiment parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# How the series is generated
MU0 = 0.0          # mean BEFORE the change
MU1 = 1.0          # mean AFTER the change
SIGMA = 1.0        # noise level
THETA = 200        # true change time
N = 400            # length of the series with a change
N_FA = 2000        # length of the no-change series (for TFA)
SEED = 42

# Detector parameters
LAMBDA = 0.2       # EWMA smoothing weight (small => smoother, slower)
L = 3.0            # limit multiplier for the EWMA chart

rng = np.random.default_rng(SEED)
print(f"mu0 = {MU0}, mu1 = {MU1}, sigma = {SIGMA}, lambda = {LAMBDA}, L = {L}")


## 1. Generate a series with a change point

Same setup as usual:

- for $t < \theta$: $x_t \sim \mathcal{N}(\mu_0, \sigma^2)$
- for $t \ge \theta$: $x_t \sim \mathcal{N}(\mu_1, \sigma^2)$

In [ ]:
def generate_mean_shift(
    n,
    theta,
    mu0=MU0,
    mu1=MU1,
    sigma=SIGMA,
    rng=None,
):
    """Gaussian series with a mean shift at time theta."""
    rng = rng or np.random.default_rng()
    x = # TODO: sample n points from N(mu0, sigma^2)
    x[theta:] = # TODO: overwrite the post-change part with N(mu1, sigma^2)
    return x


x = generate_mean_shift(N, THETA, rng=rng)

fig, ax = plt.subplots(figsize=(9, 2.8), dpi=140)
ax.plot(x, color="#1B1B1E", lw=1.0)
ax.axvline(THETA, color="#2f6f6f", lw=1.6, label=r"$\theta$ (true change)")
ax.set_xlabel("time t")
ax.set_ylabel(r"$x_t$")
ax.set_title("Series with a mean shift")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=0.25, ls="--")
plt.tight_layout()
plt.show()


## 2. Implement the EWMA chart

You need three pieces:

1. the recursion for $z_t$;
2. the steady-state width $\sigma_z = \sigma\sqrt{\lambda/(2-\lambda)}$;
3. the alarm rule $z_t > \mu_0 + L\sigma_z$.

In [ ]:
def ewma(x, mu0=MU0, sigma=SIGMA, lam=LAMBDA, L=L):
    """
    One-sided EWMA chart for an upward mean shift.

    Returns
    -------
    z   : EWMA statistic over time
    tau : first alarm index, or None
    """
    n = len(x)
    z = # TODO: allocate / initialize the EWMA sequence
    tau = None
    # TODO: steady-state standard deviation of the EWMA statistic
    sigma_z = # TODO
    threshold = # TODO: upper control limit for z_t
    # TODO: EWMA recursion + alarm rule
    return z, tau


z, tau = ewma(x)
sigma_z = SIGMA * np.sqrt(LAMBDA / (2.0 - LAMBDA))
threshold = MU0 + L * sigma_z
print(f"true change time theta = {THETA}")
print(f"alarm time         tau = {tau}")
print(f"UCL = {threshold:.3f}")


## 3. Two metrics: delay and time to false alarm

### Detection delay
On a series **with** a change:

- if $\tau \ge \theta$: $\mathrm{delay} = \tau - \theta$;
- if $\tau < \theta$: false alarm before the change;
- if there is no alarm: miss.

### Time to false alarm (TFA)
On a quiet series with mean $\mu_0$ forever, TFA is the waiting time until the first false alarm.  
If nothing happens before the horizon ends, the run is **censored**.

In [ ]:
def detection_delay(theta, tau):
    """Summarize one run on a series with a change."""
    if tau is None:
        return {"outcome": "miss", "delay": None, "false_alarm_before_cp": False}
    if tau < theta:
        return {"outcome": "false_alarm", "delay": None, "false_alarm_before_cp": True}
    return {"outcome": "detect", "delay": int(tau - theta), "false_alarm_before_cp": False}


def time_to_false_alarm(detector, n=N_FA, mu0=MU0, sigma=SIGMA, rng=None, **det_kw):
    """TFA on a series with no change."""
    rng = rng or np.random.default_rng()
    x_fa = rng.normal(mu0, sigma, size=n)
    _, tau_fa = detector(x_fa, **det_kw)
    return {
        "tfa": tau_fa,                 # None = no alarm within horizon n
        "censored": tau_fa is None,
        "horizon": n,
    }


delay_info = detection_delay(THETA, tau)
tfa_info = time_to_false_alarm(
    ewma, rng=np.random.default_rng(SEED + 1), lam=LAMBDA, L=L
)

print("Result on series with a change:", delay_info)
print("Result on series with no change:", tfa_info)


## 4. Look at the run

Top: the series.  
Bottom: the EWMA path $z_t$ and its UCL.  
If detection happens after $\theta$, we mark the delay.

In [ ]:
def plot_run(x, stat, theta, tau, threshold, title):
    t = np.arange(len(x))
    fig, axes = plt.subplots(
        2, 1, figsize=(9, 4.6), dpi=140, sharex=True,
        gridspec_kw={"hspace": 0.12},
    )

    ax = axes[0]
    ax.plot(t, x, color="#1B1B1E", lw=1.0)
    ax.axvline(theta, color="#2f6f6f", lw=1.5, label=r"$\theta$")
    if tau is not None:
        ax.axvline(tau, color="#C0392B", lw=1.4, ls="--", label=r"alarm $\tau$")
    ax.set_ylabel(r"$x_t$")
    ax.set_title(title, loc="left", fontsize=11)
    ax.legend(frameon=False, loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.25, ls="--")

    ax = axes[1]
    ax.plot(t, stat, color="#1B1B1E", lw=1.1, label='$z_t$')
    ax.axhline(threshold, color="#AAC508", lw=1.6, label='UCL')
    ax.axvline(theta, color="#2f6f6f", lw=1.5)
    if tau is not None:
        ax.axvline(tau, color="#C0392B", lw=1.4, ls="--")
        ax.scatter([tau], [stat[tau]], color="#C0392B", s=35, zorder=5)
    ax.set_xlabel("time t")
    ax.set_ylabel('$z_t$')
    ax.legend(frameon=False, loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.25, ls="--")

    info = detection_delay(theta, tau)
    if info["outcome"] == "detect":
        ymin, ymax = ax.get_ylim()
        span = ymax - ymin
        # lift the axis so the delay mark sits above the curve
        ax.set_ylim(ymin, ymax + 0.28 * span)
        y = ymax + 0.10 * span
        ax.annotate(
            "",
            xy=(tau, y),
            xytext=(theta, y),
            arrowprops=dict(arrowstyle="<->", color="#AAC508", lw=1.6),
            zorder=6,
        )
        ax.text(
            (theta + tau) / 2,
            y + 0.05 * span,
            f"delay = {info['delay']}",
            ha="center",
            va="bottom",
            color="#5C6B04",
            fontsize=9,
            fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.9, pad=1.2),
            zorder=7,
        )

    plt.show()
    return info


plot_run(x, z, THETA, tau, threshold, title="EWMA")


## 5. Average over many runs (Monte Carlo)

One series is noisy. Repeat the experiment $M$ times and look at average delay / TFA.

In [ ]:
def monte_carlo(M=200, seed=0, **det_kw):
    rng = np.random.default_rng(seed)
    delays = []
    fa_before = 0
    misses = 0
    tfas = []
    tfa_censored = 0

    for _ in range(M):
        # series WITH a change
        x_i = generate_mean_shift(N, THETA, rng=rng)
        _, tau_i = ewma(x_i, lam=det_kw.get('lam', LAMBDA), L=det_kw.get('L', L))
        info = detection_delay(THETA, tau_i)
        if info["outcome"] == "detect":
            delays.append(info["delay"])
        elif info["outcome"] == "false_alarm":
            fa_before += 1
        else:
            misses += 1

        # series WITHOUT a change
        x_fa = rng.normal(MU0, SIGMA, size=N_FA)
        _, tau_fa = ewma(x_fa, lam=det_kw.get('lam', LAMBDA), L=det_kw.get('L', L))
        if tau_fa is None:
            tfa_censored += 1
        else:
            tfas.append(tau_fa)

    return {
        "M": M,
        "mean_delay": float(np.mean(delays)) if delays else None,
        "median_delay": float(np.median(delays)) if delays else None,
        "n_detect": len(delays),
        "n_false_alarm_before_cp": fa_before,
        "n_miss": misses,
        "mean_tfa": float(np.mean(tfas)) if tfas else None,
        "median_tfa": float(np.median(tfas)) if tfas else None,
        "n_tfa_observed": len(tfas),
        "n_tfa_censored": tfa_censored,
        **det_kw,
    }


stats = monte_carlo(M=200, seed=7, lam=LAMBDA, L=L)
for key, val in stats.items():
    print(f"{key:28s}: {val}")


## 6. Play with $\lambda$ and $L$

Two knobs matter:

- $\lambda$: how much weight goes to the newest point;
- $L$: how strict the threshold is.

Try a small grid and watch delay vs TFA move together.

In [ ]:
# TODO: pick a few (lambda, L) pairs to compare
grid = # TODO
rows = [monte_carlo(M=120, seed=11, **kw) for kw in grid]

print(f"{'lam':>6}  {'L':>6}  {'mean_delay':>12}  {'mean_tfa':>12}  {'FA_before':>10}  {'miss':>6}")
for kw, r in zip(grid, rows):
    md_ = f"{r['mean_delay']:.2f}" if r["mean_delay"] is not None else "—"
    mt_ = f"{r['mean_tfa']:.1f}" if r["mean_tfa"] is not None else "censored"
    print(
        f"{kw['lam']:6.1f}  {kw['L']:6.1f}  {md_:>12}  {mt_:>12}  "
        f"{r['n_false_alarm_before_cp']:10d}  {r['n_miss']:6d}"
    )


## 7. See the trade-off on a plot

Now freeze $\lambda$ and only move the threshold $L$. Plot:

- mean detection delay vs $L$;
- mean time to false alarm vs $L$.

This makes the delay ↔ false-alarm trade-off easy to see.

In [ ]:
# TODO: fix lambda (use LAMBDA), pick a few L values,
# run monte_carlo for each, then plot mean delay and mean TFA vs L.

Ls = # TODO: e.g. [2.0, 2.5, 3.0, 3.5]
rows = # TODO: [monte_carlo(M=120, seed=13, lam=LAMBDA, L=L_i) for L_i in Ls]

fig, ax1 = plt.subplots(figsize=(7.2, 3.2), dpi=140)

delays = # TODO: list of mean_delay from rows
tfas = # TODO: list of mean_tfa from rows  (use np.nan if censored / None)

ax1.plot(Ls, delays, "o-", color="#2f6f6f", lw=1.8, label="mean delay")
ax1.set_xlabel(r"limit multiplier $L$")
ax1.set_ylabel("mean detection delay", color="#2f6f6f")
ax1.tick_params(axis="y", labelcolor="#2f6f6f")

ax2 = ax1.twinx()
# TODO: plot mean TFA on ax2 (suggestion: "s--" and color "#AAC508")
ax2.set_ylabel("mean time to false alarm", color="#5C6B04")
ax2.tick_params(axis="y", labelcolor="#5C6B04")

ax1.set_title("Delay vs TFA as $L$ changes (fixed lambda)", loc="left", fontsize=11)
ax1.grid(True, alpha=0.25, ls="--")
ax1.spines["top"].set_visible(False)

# TODO: add a combined legend for both curves

plt.tight_layout()
plt.show()


## 8. Harder data: the friendly jump was too easy

So far the post-change mean was $\mu_1 = 1$ with $\sigma = 1$ — a loud shift.  
Real changes are often quieter. Keep the **same detector settings** and try two nastier worlds.

**A. Small mean shift**  
Use $\mu_1 = 0.4$ instead of $1.0$. The change is still there, but each single point rarely clears a Shewhart-style bar by itself.

**B. Variance change, mean fixed**  
Keep mean $\mu_0$ forever, but after $\theta$ jump the noise: $\sigma_0 \to \sigma_1$ (e.g. $1 \to 2$).  
Your chart was built to watch the *level*. Does it still make sense? What do you observe?

For each scenario, report delay / misses (Monte Carlo is enough) and write 2–3 sentences on what broke.

In [ ]:
# TODO: implement a variance-shift generator (mean fixed, sigma changes after theta)

def generate_variance_shift(
    n, theta, mu=MU0, sigma0=SIGMA, sigma1=2.0, rng=None
):
    """Same mean, larger noise after theta."""
    rng = rng or np.random.default_rng()
    x = # TODO
    x[theta:] = # TODO
    return x


def summarize_scenario(name, make_series, M=150, seed=21):
    """Run the detector once for a plot, then Monte Carlo delay / miss counts."""
    rng = np.random.default_rng(seed)
    delays, misses, fa_before = [], 0, 0

    x_hard = make_series(rng)
    z_h, tau_h = ewma(x_hard)
    info_one = detection_delay(THETA, tau_h)

    for _ in range(M):
        x_i = make_series(rng)
        _, tau_i = ewma(x_i)
        info = detection_delay(THETA, tau_i)
        # TODO: update delays / fa_before / misses from info
        pass

    print(f"=== {name} ===")
    print("one-run:", info_one)
    # TODO: print MC summary (mean delay, #detect, #FA_before, #miss)
    return x_hard, tau_h, info_one


# A) small mean shift
x_small, tau_small, _ = summarize_scenario(
    "small mean shift (mu1=0.4)",
    make_series=lambda rng: generate_mean_shift(
        N, THETA, mu0=MU0, mu1=0.4, sigma=SIGMA, rng=rng
    ),
)

# TODO: plot x_small with theta and tau_small

# B) variance inflation, mean unchanged
x_var, tau_var, _ = summarize_scenario(
    "variance shift (sigma: 1 -> 2, mean fixed)",
    make_series=lambda rng: generate_variance_shift(
        N, THETA, mu=MU0, sigma0=1.0, sigma1=2.0, rng=rng
    ),
)

# TODO: plot x_var with theta and tau_var

# TODO: short write-up — what got worse vs the original mu1=1 case, and why?


## What to show in the seminar

1. A plot: the series + the monitoring statistic, with $\theta$ and $\tau$ marked.
2. Numbers: delay on one example, plus Monte Carlo averages for delay and TFA.
3. The delay / TFA figure against the threshold.
4. What happened on the **harder data** (small shift / variance change): who struggles, and why.